# Phase 1 — Falsification pilot (Gate 1)

Runs the whole Phase 1 pilot: PREREG splits → LLaVA-1.5 captions → CHAIR labels → OWLv2 + CLIP evidence under 4 image conditions → Gate 1 report.

**Before you start**
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Results are saved to Google Drive, so a disconnect loses nothing; just re-run all cells.
- **Kaggle:** Settings → Accelerator **GPU T4 x2** (one is used), **Internet ON** (needs a phone-verified account).

**Time:** about 5 min smoke test, then 1–1.5 h for the full run. Every stage resumes where it stopped.

**When it finishes:** the last cell prints the Gate 1 report and makes `phase1_results.zip`. Put that zip in your Google Drive (Colab already saves it there) or attach it in the chat.

In [ ]:
# 1. Where are we, and is there a GPU?
import os, subprocess, sys
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")
print("Colab:", ON_COLAB, "| Kaggle:", ON_KAGGLE)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "NO GPU FOUND - switch the runtime to a T4 GPU before continuing")

In [ ]:
# 2. Output folder (persistent)
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/BCRBP_phase1"
    WORK = "/content"
elif ON_KAGGLE:
    OUT = "/kaggle/working/BCRBP_phase1"
    WORK = "/kaggle/working"
else:
    OUT = os.path.abspath("BCRBP_phase1")
    WORK = os.getcwd()
os.makedirs(OUT, exist_ok=True)
print("Results go to:", OUT)

In [ ]:
# 3. Get the code (main branch) and install the few extra packages
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
BRANCH = "main"
os.chdir(WORK)
if not os.path.exists("bcrbp"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "bcrbp"], check=True)
else:
    subprocess.run(["git", "-C", "bcrbp", "pull", "--ff-only"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())
!git log --oneline -1

In [ ]:
# 4. Smoke test on 4 images (~5 min). Fix any error here before the full run.
!python scripts/phase1_pilot.py fetch    --out "$OUT"
!python scripts/phase1_pilot.py prepare  --out "$OUT"
!python scripts/phase1_pilot.py captions --out "$OUT" --limit 4
!python scripts/phase1_pilot.py evidence --out "$OUT" --limit 4
import json
caps = [json.loads(l) for l in open(f"{OUT}/captions.jsonl")]
print(len(caps), "captions so far. Example:\n", caps[0]["caption"][:600])

In [ ]:
# 5. Full run: 200 pilot images (about 1-1.5 h). Safe to re-run after a disconnect.
!python scripts/phase1_pilot.py captions --out "$OUT"
!python scripts/phase1_pilot.py evidence --out "$OUT"

In [ ]:
# 6. Gate 1 analysis (CPU, ~5-10 min) and report
!python scripts/phase1_pilot.py analyze --out "$OUT"
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/gate1_report.md").read()))

In [ ]:
# 7. Package results (no images, no COCO annotations) for sharing
import zipfile
keep = ["manifest.json", "gt_objects.json", "captions.jsonl", "evidence.jsonl",
        "claims_table.jsonl", "gate1_results.json", "gate1_report.md"]
zpath = os.path.join(OUT, "phase1_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        p = os.path.join(OUT, f)
        if os.path.exists(p):
            z.write(p, f)
    z.writestr("environment.txt", f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("Saved:", zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB")
if ON_KAGGLE:
    print("Kaggle: download it from the Output panel (right side) -> BCRBP_phase1/phase1_results.zip")